# 🔍 **EDA (Exploratory Data Analysis)**

**Датасет:** Бинарное прогнозирование срыва сроков доставки еды (SLA)  
**Бизнес-задача:** Требуется классифицировать риск задержки доставки заказа более чем на 5 минут от обещанного времени на этапе оформления, используя параметры ресторана, рейтинг курьера, погодные условия и пробки.  

---

## 📚 **Импорты и настройки**

In [12]:
# Основные библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

# Загрузка данных
import kagglehub

## **1. Загрузка данных**

In [13]:
# Загрузка датасета с Kaggle
path = kagglehub.dataset_download("razanihababdellatif/food-delivery-orders-and-eta-logistics-dataset")
df = pd.read_csv(f"{path}/food_delivery_orders_dataset.csv", index_col=0)

Using Colab cache for faster access to the 'food-delivery-orders-and-eta-logistics-dataset' dataset.


---

# **2. Первичный осмотр данных**

## **2.1 Первый взгляд на данные**

In [14]:
# Первые 5 строк
df.head()

,customer_id,restaurant_id,driver_id,order_timestamp,order_date,order_hour,day_of_week,is_weekend,city,delivery_area,...,traffic_level,delivery_partner_experience_months,delivery_partner_rating,restaurant_preparation_time_minutes,estimated_delivery_time_minutes,actual_delivery_time_minutes,late_delivery,order_status,cancellation_reason,customer_rating
order_id,,,,,,,,,,,,,,,,,,,,,
ORD_000001,CUS_00001,RES_0008,DRV_0075,2025-10-05 15:54:00,2025-10-05,15,Sunday,1,City_C,City_C_Area_3,...,Medium,14,4.4,25,25,29.0,0.0,Completed,NaN,4.6
ORD_000002,CUS_00013,RES_0024,DRV_0131,2025-10-01 20:00:00,2025-10-01,20,Wednesday,0,City_A,City_A_Area_1,...,High,44,4.7,12,20,24.0,0.0,Completed,NaN,5.0
ORD_000003,CUS_00049,RES_0013,DRV_0087,2025-10-07 18:24:00,2025-10-07,18,Tuesday,0,City_C,City_C_Area_3,...,High,1,5.0,11,63,72.0,1.0,Completed,NaN,3.6
ORD_000004,CUS_00007,RES_0001,DRV_0177,2025-10-30 16:54:00,2025-10-30,16,Thursday,0,City_A,City_A_Area_1,...,Low,50,4.3,12,23,31.0,1.0,Completed,NaN,4.1
ORD_000005,CUS_00325,RES_0029,DRV_0114,2025-10-15 18:34:00,2025-10-15,18,Wednesday,0,City_B,City_B_Area_6,...,Medium,21,4.0,9,32,42.0,1.0,Completed,NaN,3.9


In [15]:
# Случайная выборка
df.sample(5, random_state=42)

,customer_id,restaurant_id,driver_id,order_timestamp,order_date,order_hour,day_of_week,is_weekend,city,delivery_area,...,traffic_level,delivery_partner_experience_months,delivery_partner_rating,restaurant_preparation_time_minutes,estimated_delivery_time_minutes,actual_delivery_time_minutes,late_delivery,order_status,cancellation_reason,customer_rating
order_id,,,,,,,,,,,,,,,,,,,,,
ORD_033554,CUS_00002,RES_0026,DRV_0195,2025-10-16 03:25:00,2025-10-16,3,Thursday,0,City_C,City_C_Area_4,...,Low,15,4.7,12,15,16.0,0.0,Completed,NaN,4.7
ORD_009428,CUS_00006,RES_0009,DRV_0033,2025-10-02 13:10:00,2025-10-02,13,Thursday,0,City_B,City_B_Area_10,...,Low,28,4.3,18,31,33.0,0.0,Completed,NaN,5.0
ORD_000200,CUS_00186,RES_0036,DRV_0031,2025-10-18 14:27:00,2025-10-18,14,Saturday,1,City_C,City_C_Area_5,...,Medium,25,5.0,10,38,40.0,0.0,Completed,NaN,4.6
ORD_012448,CUS_00001,RES_0119,DRV_0064,2025-10-07 12:46:00,2025-10-07,12,Tuesday,0,City_C,City_C_Area_5,...,Medium,29,4.3,10,19,26.0,1.0,Completed,NaN,3.6
ORD_039490,CUS_00076,RES_0005,DRV_0192,2025-10-04 09:10:00,2025-10-04,9,Saturday,1,City_A,City_A_Area_4,...,Medium,6,4.8,16,26,31.0,0.0,Completed,NaN,4.4


## **2.2 Технический паспорт данных**

In [16]:
# Информация о структуре
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50000 entries, ORD_000001 to ORD_050000
Data columns (total 36 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   customer_id                          50000 non-null  object 
 1   restaurant_id                        50000 non-null  object 
 2   driver_id                            50000 non-null  object 
 3   order_timestamp                      50000 non-null  object 
 4   order_date                           50000 non-null  object 
 5   order_hour                           50000 non-null  int64  
 6   day_of_week                          50000 non-null  object 
 7   is_weekend                           50000 non-null  int64  
 8   city                                 50000 non-null  object 
 9   delivery_area                        50000 non-null  object 
 10  customer_age                         50000 non-null  int64  
 11  customer_type      

### Анализ `.info()`:

| Метрика | Значение | Вывод |
|---------|----------|-------|
| **Типы данных** | float64(9), int64(6), object(5) | OK, но можно оптимизировать |
| **Пропуски** | Есть в `artists`, `album_name`, `track_name` | Нужно исследовать |

## **2.3 Оптимизация памяти**

Теперь, когда данные очищены, можно оптимизировать типы данных для экономии памяти.

In [17]:
# Замер памяти до оптимизации
memory_before = df.memory_usage(deep=True).sum() / 1024**2
print(f"💾 Память до: {memory_before:.2f} MB")

💾 Память до: 53.23 MB


In [18]:
# Анализ типов данных
df.dtypes.value_counts()

,count
object,16
float64,12
int64,8


In [19]:
# Числовые типы
# выбрать атоматически
int_cols = df.select_dtypes(include=['int64']).columns
float_cols = df.select_dtypes(include=['float64']).columns

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast='float')

# Категориальные типы (уникальность < 20%)
cat_candidates = ['explicit', 'track_genre', 'artists', 'album_name']
for col in cat_candidates:
    if col in df.columns:
        unique_ratio = df[col].nunique() / len(df)
        if unique_ratio < 0.2:
            df[col] = df[col].astype('category')
            print(f"📦 {col} → category (уникальность: {unique_ratio:.1%})")

# Замер памяти после оптимизации
memory_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"\n💾 Память после: {memory_after:.2f} MB")
print(f"📉 Экономия: {(1 - memory_after/memory_before)*100:.1f}%")


💾 Память после: 48.32 MB
📉 Экономия: 9.2%


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50000 entries, ORD_000001 to ORD_050000
Data columns (total 36 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   customer_id                          50000 non-null  object 
 1   restaurant_id                        50000 non-null  object 
 2   driver_id                            50000 non-null  object 
 3   order_timestamp                      50000 non-null  object 
 4   order_date                           50000 non-null  object 
 5   order_hour                           50000 non-null  int8   
 6   day_of_week                          50000 non-null  object 
 7   is_weekend                           50000 non-null  int8   
 8   city                                 50000 non-null  object 
 9   delivery_area                        50000 non-null  object 
 10  customer_age                         50000 non-null  int8   
 11  customer_type      

## **2.4 Статистическая сводка**

In [21]:
# Числовые переменные
df.describe().T

,count,mean,std,min,25%,50%,75%,max
order_hour,50000.0,14.465900,5.231155,0.00,11.00,14.000,19.000000,23.000000
is_weekend,50000.0,0.267120,0.442460,0.00,0.00,0.000,1.000000,1.000000
customer_age,50000.0,27.686580,7.537796,18.00,21.00,27.000,33.000000,55.000000
restaurant_rating,50000.0,4.207956,0.450923,2.80,3.90,4.200,4.600000,5.000000
items_count,50000.0,2.910020,2.029070,1.00,1.00,2.000,4.000000,10.000000
subtotal,50000.0,30.897938,27.068874,5.10,13.10,22.825,40.119999,362.980011
discount_percent,50000.0,4.495700,6.430008,0.00,0.00,0.000,10.000000,20.000000
tax_amount,50000.0,4.128796,3.636319,0.57,1.73,2.890,5.390000,50.820000
service_fee,50000.0,1.474600,1.378104,0.13,0.58,1.040,1.890000,25.209999
delivery_fee,50000.0,3.075108,2.298987,0.00,0.00,3.500,4.500000,12.000000


In [22]:
# Категориальные переменные
df.describe(include=['object', 'category']).T

,count,unique,top,freq
customer_id,50000,4979,CUS_00001,4510
restaurant_id,50000,250,RES_0001,8837
driver_id,50000,200,DRV_0066,315
order_timestamp,50000,25459,2025-10-11 19:51:00,9
order_date,50000,30,2025-10-23,1741
day_of_week,50000,7,Thursday,8472
city,50000,4,City_A,19888
delivery_area,50000,40,City_A_Area_1,2056
customer_type,50000,3,Regular,25098
restaurant_type,50000,4,Fast Food,27593


---

# **3. Автоматизация EDA с ydata-profiling** 🚀

In [23]:
# Автоматический EDA

#!pip install ydata-profiling
!pip install fg-data-profiling

from ydata_profiling import ProfileReport

## **3.1 Базовое использование**

In [ ]:
# Генерация отчёта (это может занять 1-2 минуты)

profile = ProfileReport(
    df,
    title="Spotify Tracks EDA Report",
    explorative=True,
    correlations={
    "auto": {"calculate": False},
    "phi_k": {"calculate": True}
    }
)
profile




Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


 22%|██▏       | 8/36 [00:01<00:05,  5.26it/s]


In [ ]:
# Сохранение в HTML
profile.to_file("spotify_SLA_report.html")

**ydata-profiling — это инструмент для экономии времени, но не замена пониманию данных!**